# <center>Own EDA<center> 
<br>
<br>

### Background: <br>
The data source is https://www.netflix.com/tudum/top10 <br>
We downloaded 3 xlsx files named global_weekly, globall_alltime and country_weekly that we merged into netflix.csv and the EDA is based on it.

In [1]:
import duckdb
import pandas as pd

### Creating a dataframe and checking few rows

In [2]:
df = pd.read_csv("../data/processed/netflix.csv")
df.head(3)

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0
1,1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,NaN,2,Films (Non-English),10.0,1600000.0,115.998,800000.0,2.0
2,2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458338 entries, 0 to 458337
Data columns (total 15 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Unnamed: 0                          458338 non-null  int64  
 1   country_name                        458338 non-null  object 
 2   country_iso2                        458338 non-null  object 
 3   week                                458338 non-null  object 
 4   country_category                    458338 non-null  object 
 5   country_weekly_rank                 458338 non-null  int64  
 6   show_title                          458338 non-null  object 
 7   season_title                        223904 non-null  object 
 8   country_cumulative_weeks_in_top_10  458338 non-null  int64  
 9   global_category                     306075 non-null  object 
 10  global_weekly_rank                  306075 non-null  float64
 11  global_weekly_hours_viewed

In [4]:
df.columns.tolist()

['Unnamed: 0',
 'country_name',
 'country_iso2',
 'week',
 'country_category',
 'country_weekly_rank',
 'show_title',
 'season_title',
 'country_cumulative_weeks_in_top_10',
 'global_category',
 'global_weekly_rank',
 'global_weekly_hours_viewed',
 'runtime',
 'global_weekly_views',
 'global_cumulative_weeks_in_top_10']

##### "week" is already datetime64, so no conversion needed during cleaning!



In [5]:
# --- Suppressing unwanted "Unnamed" column ---
df = df.drop(columns=["Unnamed: 0"], errors="ignore")  # drop it right away
df.head(1)

,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0


### Counting numbers of raws and columns


In [6]:
df.shape
print(f"File has respectively: raws, columns: {df.shape}")

File has respectively: raws, columns: (458338, 14)


### Checking the columns

In [7]:
# --- Viewing the columns ---
df.columns

Index(['country_name', 'country_iso2', 'week', 'country_category',
       'country_weekly_rank', 'show_title', 'season_title',
       'country_cumulative_weeks_in_top_10', 'global_category',
       'global_weekly_rank', 'global_weekly_hours_viewed', 'runtime',
       'global_weekly_views', 'global_cumulative_weeks_in_top_10'],
      dtype='object')

### Note: No beneficial to use .describe() as the most interesting columns are categorical "country_name", "show_title" and "week"

In [8]:
# --- Uncomment below if you need to see it ---
# df.describe()

### Distribution films Vs series

In [9]:
df["country_category"].value_counts()

country_category
Films    229208
TV       229130
Name: count, dtype: int64

### How many countries are in the dataset?

In [10]:
# --- Unique country count ---
df["country_name"].nunique()


94

In [11]:
# --- Country and their number of rows ---
df["country_name"].value_counts()

country_name
Argentina               4922
Bahamas                 4922
Chile                   4922
Colombia                4922
Brazil                  4922
                        ... 
Sweden                  4920
United States           4920
United Arab Emirates    4920
Vietnam                 4920
Russia                   700
Name: count, Length: 94, dtype: int64

### The Russian case, why 700 rows?

In [12]:
# --- Seeing the registered weeks ---
russia_weeks = df[df["country_name"] == "Russia"]["week"].value_counts()
print(f"The registered weeks (and years) for Russia are: {russia_weeks}")

The registered weeks (and years) for Russia are: week
2022-02-27    20
2022-02-20    20
2022-02-13    20
2022-02-06    20
2022-01-30    20
2022-01-23    20
2022-01-16    20
2022-01-09    20
2022-01-02    20
2021-12-26    20
2021-12-19    20
2021-12-12    20
2021-12-05    20
2021-11-28    20
2021-11-21    20
2021-11-14    20
2021-11-07    20
2021-10-31    20
2021-10-24    20
2021-10-17    20
2021-10-10    20
2021-10-03    20
2021-09-26    20
2021-09-19    20
2021-09-12    20
2021-09-05    20
2021-08-29    20
2021-08-22    20
2021-08-15    20
2021-08-08    20
2021-08-01    20
2021-07-25    20
2021-07-18    20
2021-07-11    20
2021-07-04    20
Name: count, dtype: int64


In [13]:
# --- Confirming the last week for Russia ---
df[df["country_name"] == "Russia"]["week"].max()

'2022-02-27'

### Dataset range (dates)

In [14]:
df['week'] = pd.to_datetime(df['week'])
year_counts = df['week'].dt.year.value_counts()
year_counts.sort_index(ascending=False)

week
2026    20460
2025    96720
2024    96798
2023    98580
2022    96900
2021    48880
Name: count, dtype: int64

In [15]:
# --- First date and the last of the dataset ---
df["week"].min(), df["week"].max()

(Timestamp('2021-07-04 00:00:00'), Timestamp('2026-03-15 00:00:00'))

### How many nulls?

In [16]:
df.isnull().sum()

country_name                               0
country_iso2                               0
week                                       0
country_category                           0
country_weekly_rank                        0
show_title                                 0
season_title                          234434
country_cumulative_weeks_in_top_10         0
global_category                       152263
global_weekly_rank                    152263
global_weekly_hours_viewed            152263
runtime                               277665
global_weekly_views                   277665
global_cumulative_weeks_in_top_10     152263
dtype: int64

The nulls counted above are not errors but structural.<br>
For example, the "season_title" column contains many NaN because movies have no season. Another example is for column "global_weekly_rank", if a film only charted locally in one country but not globally, there is no global rank to record it.<br>
The important nulls would be the ones in the columns: "country_name", "show_title" or "week".<br>
So I will just replace some of them with "Not applciable" as N/A in the cleaning section below.

### Focus on the Nordics (as the demos with UX will focus on it)

In [17]:
# --- Creating a df called df_nordics to use with duckdb ---
duckdb.register("df_nordics", df)

In [18]:
# --- Checking what Nordic countries we have ---
duckdb.sql("""
SELECT 
    country_name, COUNT(*) as rows
FROM 
    df_nordics
WHERE 
    country_name IN ('Iceland', 'Norway', 'Sweden', 'Finland', 'Denmark')
GROUP BY country_name
ORDER BY rows DESC
""").df()

,country_name,rows
0,Finland,4921
1,Iceland,4921
2,Denmark,4920
3,Norway,4920
4,Sweden,4920


In [19]:
#--- Comparing starting and ending dates for the Nordic countries ---
duckdb.sql("""
SELECT 
    country_name, 
    MIN(week) AS start, 
    MAX(week) AS end, 
    COUNT(DISTINCT week) AS weeks
FROM 
    df_nordics
WHERE 
    country_name IN ('Iceland', 'Norway', 'Sweden', 'Finland', 'Denmark')
GROUP BY country_name
ORDER BY country_name
""").df()

,country_name,start,end,weeks
0,Denmark,2021-07-04,2026-03-15,246
1,Finland,2021-07-04,2026-03-15,246
2,Iceland,2021-07-04,2026-03-15,246
3,Norway,2021-07-04,2026-03-15,246
4,Sweden,2021-07-04,2026-03-15,246


In [20]:
# --- Films/series that spent the most time in the top 10 in Sweden in 2025 ---
duckdb.sql("""
SELECT 
    show_title, 
    COUNT(*) AS weeks_in_top10_2025
FROM 
    df_nordics
WHERE 
    country_name = 'Sweden'
AND YEAR(week) = 2025
GROUP BY show_title
ORDER BY weeks_in_top10_2025 DESC
LIMIT 10
""").df()

,show_title,weeks_in_top10_2025
0,KPop Demon Hunters,26
1,Squid Game,16
2,Dept. Q,13
3,Wednesday,12
4,Barracuda Queens,12
5,Adolescence,11
6,Stranger Things,11
7,Sirens,11
8,Love Is Blind,10
9,Love is Blind: Sweden,9


In [21]:
# --- Films and series distribution per nordic country ---
duckdb.sql("""
SELECT country_name, country_category, COUNT(*) as count
FROM df_nordics
WHERE country_name IN ('Iceland', 'Norway', 'Sweden', 'Finland', 'Denmark')
GROUP BY country_name, country_category
ORDER BY country_name
""").df()

,country_name,country_category,count
0,Denmark,Films,2460
1,Denmark,TV,2460
2,Finland,Films,2461
3,Finland,TV,2460
4,Iceland,Films,2461
5,Iceland,TV,2460
6,Norway,Films,2460
7,Norway,TV,2460
8,Sweden,Films,2460
9,Sweden,TV,2460


## <center>Data cleaning<center>

In [22]:
df.head(1)

,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0


### Any duplicates?

In [23]:
# Counting the duplicates
df.duplicated().sum()

np.int64(0)

### The NaN question

In [24]:
# --- Distinction between the columns with numbers and the ones having text.
# --- I keep NaN in the numeric columns to avoid possible issues with calculations ---
# --- In the text columns, I replace NaN ---> N/A for "Not applicable" 
text_columns = ["season_title", "global_category"]
df[text_columns] = df[text_columns].fillna("N/A")
df.head(3)

,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,N/A,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0
1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,N/A,2,Films (Non-English),10.0,1600000.0,115.998,800000.0,2.0
2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,N/A,1,N/A,NaN,NaN,NaN,NaN,NaN


### Renaming columns: country_iso2 and country_category

In [25]:
# --- Renaming ---
df = duckdb.sql("""---sql
SELECT * RENAME (
    country_iso2 AS country_code, 
    country_category AS film_or_tv_series,
    runtime AS duration_minutes)
FROM 
    df
""").df()
df.head(1)

,country_name,country_code,week,film_or_tv_series,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,duration_minutes,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,N/A,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0


### Rounding the runtime column

In [26]:
# NaN --> 0 | rounding | float --> integer
df['duration_minutes'] = df['duration_minutes'].fillna(0).round(0).astype(int)
df.head(2)

,country_name,country_code,week,film_or_tv_series,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,duration_minutes,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,N/A,2,Films (English),1.0,80600000.0,109,44400000.0,2.0
1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,N/A,2,Films (Non-English),10.0,1600000.0,116,800000.0,2.0


### The use of .str.lower() to distinguish

In [27]:
# As some of the data/columns will be displayed in visuals I would keep the "country_name", "country_codes" and "season_title" with a capital letter.
# --- 2 columns can be in small letters ---
df["film_or_tv_series"] = df["film_or_tv_series"].str.lower()
df["global_category"] = df["global_category"].str.lower()
df.head(1)


,country_name,country_code,week,film_or_tv_series,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,duration_minutes,global_weekly_views,global_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,films,1,War Machine,N/A,2,films (english),1.0,80600000.0,109,44400000.0,2.0


## <center>Exporting the cleaned dataset<center>

In [ ]:
df.to_csv(r"C:\Users\girau\Documents\G9\Netflix_Analytics_DE_UX\data\processed\netflix_cleaned.csv", index=False)